
# Buy Fee Model V10.1 — Minimal Success vs Failure Diagnostics

## Purpose

This is a **lightweight replacement** for V10.

It uses only the small V8.2 event-level DiD file:

`04_DID_EVENT_LEVEL_RESULTS.csv`

No:
- sklearn
- statsmodels
- matplotlib
- large weekly panel
- buyer-file join

It compares successful vs unsuccessful Buy Fee changes by:

- Location
- Price tier
- Lane
- Fee-change percentage
- Baseline conversion
- Baseline volume
- Previous Buy Fee
- Baseline RPU
- Season / quarter / month

## Balanced Success definition

An event is a Balanced Success when:

- `DiD_RPU_Change_Dollar > 0`
- `DiD_Conversion_Change_pp >= -2`
- `DiD_Volume_Change_Pct >= -0.10`

These thresholds are configurable below.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# ==============================""==============================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v10_1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Success thresholds
RPU_SUCCESS_MIN = 0.0
CONVERSION_SAFE_MIN_PP = -2.0
VOLUME_SAFE_MIN_PCT = -0.10

# Minimum rows needed to show a group in diagnostics
MIN_GROUP_EVENTS = 10

print("=" * 80)
print("V10.1 INPUT CHECK")
print("=" * 80)

print("Input file:")
print(DID_FILE)

print("\nExists:")
print(DID_FILE.exists())

if not DID_FILE.exists():
    raise FileNotFoundError(
        f"Required file not found:\n{DID_FILE}"
    )

print("\nOutput folder:")
print(OUTPUT_DIR)


V10.1 INPUT CHECK
Input file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\04_DID_EVENT_LEVEL_RESULTS.csv

Exists:
True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v10_1_outputs


## 1. Load V8.2 event-level DiD data

In [2]:

df = pd.read_csv(
    DID_FILE,
    low_memory=True
)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

required = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )

numeric_cols = [
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

for c in numeric_cols:
    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

print("\nUnique locations:", df["Treated_Location"].nunique())
print("Unique lanes:", df["Treated_Lane"].nunique())
print("Corporate change dates:", df["Corporate_Change_ID"].nunique())

display(df.head())


Rows: 1,824
Columns: 19

Unique locations: 35
Unique lanes: 100
Corporate change dates: 37


,Segment_Change_ID,Corporate_Change_ID,BUYFEE_CHANGE_AMOUNT,PREVIOUS_BUYFEE,FEE_CHANGE_PCT,Treated_Location,Treated_Lane,Treated_Price_Tier,Number_of_Controls,Median_Match_Distance,DiD_Volume_Change_Pct,DiD_Conversion_Change_pp,DiD_RPU_Change_Dollar,DiD_Fee_Pass_Through,pre_registered,pre_sold,pre_conversion,pre_rpu,pre_avg_saleamt
0,ATLANTA_A_1000-2000_2024-12-01,2024-12-01,30.0,250.0,0.120000,ATLANTA,A,1000-2000,3,1.308194,0.034685,2.020544,25.014089,0.833803,57,38,0.666667,0.000000,1468.421053
1,ATLANTA_A_10000-11000_2024-12-01,2024-12-01,50.0,460.0,0.108696,ATLANTA,A,10000-11000,3,1.505764,-0.134921,-1.339671,30.993895,0.619878,32,14,0.437500,390.000000,10218.750000
2,ATLANTA_A_2000-3000_2024-12-01,2024-12-01,40.0,325.0,0.123077,ATLANTA,A,2000-3000,3,1.029995,-0.173794,-21.944969,1.047732,0.026193,55,34,0.618182,318.088235,2323.636364
3,ATLANTA_A_3000-4000_2024-12-01,2024-12-01,40.0,350.0,0.114286,ATLANTA,A,3000-4000,3,0.912546,0.081020,-4.635448,13.116380,0.327910,56,29,0.517857,343.103448,3425.000000
4,ATLANTA_A_4000-5000_2024-12-01,2024-12-01,40.0,365.0,0.109589,ATLANTA,A,4000-5000,3,1.079346,0.253197,1.935671,56.677123,1.416928,41,17,0.414634,302.941176,4336.585366


## 2. Create seasonality and success flags

In [3]:

df["Change_Date"] = pd.to_datetime(
    df["Corporate_Change_ID"],
    errors="coerce"
)

df["Change_Year"] = df["Change_Date"].dt.year
df["Change_Month"] = df["Change_Date"].dt.month
df["Change_Month_Name"] = df["Change_Date"].dt.month_name()
df["Change_Quarter"] = df["Change_Date"].dt.quarter

def get_season(month):
    if pd.isna(month):
        return "Unknown"
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Fall"

df["Season"] = df["Change_Month"].apply(get_season)

df["RPU_Success"] = (
    df["DiD_RPU_Change_Dollar"] > RPU_SUCCESS_MIN
)

df["Conversion_Safe"] = (
    df["DiD_Conversion_Change_pp"] >= CONVERSION_SAFE_MIN_PP
)

df["Volume_Safe"] = (
    df["DiD_Volume_Change_Pct"] >= VOLUME_SAFE_MIN_PCT
)

df["Balanced_Success"] = (
    df["RPU_Success"]
    & df["Conversion_Safe"]
    & df["Volume_Safe"]
)

print("Balanced success rate:", round(df["Balanced_Success"].mean(), 4))
print("RPU success rate:", round(df["RPU_Success"].mean(), 4))
print("Conversion safe rate:", round(df["Conversion_Safe"].mean(), 4))
print("Volume safe rate:", round(df["Volume_Safe"].mean(), 4))


Balanced success rate: 0.2796
RPU success rate: 0.5641
Conversion safe rate: 0.5899
Volume safe rate: 0.6234


## 3. Create simple diagnostic bands

In [4]:

def safe_qcut(series, q=4):
    x = pd.to_numeric(
        series,
        errors="coerce"
    )

    valid = x.notna()

    out = pd.Series(
        "Missing",
        index=series.index,
        dtype="object"
    )

    if valid.sum() < q:
        return out

    try:
        out.loc[valid] = pd.qcut(
            x.loc[valid],
            q=q,
            duplicates="drop"
        ).astype(str)

    except Exception:
        out.loc[valid] = "Available"

    return out

df["Fee_Change_Pct_Band"] = safe_qcut(
    df["FEE_CHANGE_PCT"]
)

df["Baseline_Conversion_Band"] = safe_qcut(
    df["pre_conversion"]
)

df["Baseline_Volume_Band"] = safe_qcut(
    df["pre_sold"]
)

df["Previous_BuyFee_Band"] = safe_qcut(
    df["PREVIOUS_BUYFEE"]
)

df["Baseline_RPU_Band"] = safe_qcut(
    df["pre_rpu"]
)

df["Sale_Amount_Band"] = safe_qcut(
    df["pre_avg_saleamt"]
)

print("Diagnostic bands created.")


Diagnostic bands created.


## 4. Reusable comparison function

In [5]:

def compare_groups(data, group_col, min_events=10):

    temp = data[
        data[group_col].notna()
    ].copy()

    grouped = (
        temp.groupby(
            group_col,
            dropna=False
        )
        .agg(
            Events=("Segment_Change_ID", "count"),
            Locations=("Treated_Location", "nunique"),
            Change_Dates=("Corporate_Change_ID", "nunique"),

            Balanced_Success_Rate=("Balanced_Success", "mean"),
            RPU_Success_Rate=("RPU_Success", "mean"),
            Conversion_Safe_Rate=("Conversion_Safe", "mean"),
            Volume_Safe_Rate=("Volume_Safe", "mean"),

            Median_RPU_Effect=("DiD_RPU_Change_Dollar", "median"),
            Median_Conversion_Effect_pp=("DiD_Conversion_Change_pp", "median"),
            Median_Volume_Effect_Pct=("DiD_Volume_Change_Pct", "median"),

            Median_Fee_Change=("BUYFEE_CHANGE_AMOUNT", "median"),
            Median_Fee_Change_Pct=("FEE_CHANGE_PCT", "median"),

            Median_Baseline_Conversion=("pre_conversion", "median"),
            Median_Baseline_Sold=("pre_sold", "median"),
            Median_Previous_BuyFee=("PREVIOUS_BUYFEE", "median"),
            Median_Baseline_RPU=("pre_rpu", "median"),
            Median_Sale_Amount=("pre_avg_saleamt", "median"),
        )
        .reset_index()
    )

    grouped = grouped[
        grouped["Events"] >= min_events
    ].copy()

    grouped = grouped.sort_values(
        ["Balanced_Success_Rate", "Events"],
        ascending=[False, False]
    )

    return grouped


## 5. Run all requested diagnostics

In [6]:

diagnostics = {}

diagnostics["LOCATION"] = compare_groups(
    df,
    "Treated_Location",
    MIN_GROUP_EVENTS
)

diagnostics["PRICE_TIER"] = compare_groups(
    df,
    "Treated_Price_Tier",
    MIN_GROUP_EVENTS
)

diagnostics["LANE"] = compare_groups(
    df,
    "Treated_Lane",
    MIN_GROUP_EVENTS
)

diagnostics["FEE_CHANGE_PCT"] = compare_groups(
    df,
    "Fee_Change_Pct_Band",
    MIN_GROUP_EVENTS
)

diagnostics["BASELINE_CONVERSION"] = compare_groups(
    df,
    "Baseline_Conversion_Band",
    MIN_GROUP_EVENTS
)

diagnostics["BASELINE_VOLUME"] = compare_groups(
    df,
    "Baseline_Volume_Band",
    MIN_GROUP_EVENTS
)

diagnostics["PREVIOUS_BUYFEE"] = compare_groups(
    df,
    "Previous_BuyFee_Band",
    MIN_GROUP_EVENTS
)

diagnostics["BASELINE_RPU"] = compare_groups(
    df,
    "Baseline_RPU_Band",
    MIN_GROUP_EVENTS
)

diagnostics["SALE_AMOUNT"] = compare_groups(
    df,
    "Sale_Amount_Band",
    MIN_GROUP_EVENTS
)

diagnostics["SEASON"] = compare_groups(
    df,
    "Season",
    MIN_GROUP_EVENTS
)

diagnostics["QUARTER"] = compare_groups(
    df,
    "Change_Quarter",
    MIN_GROUP_EVENTS
)

diagnostics["MONTH"] = compare_groups(
    df,
    "Change_Month_Name",
    MIN_GROUP_EVENTS
)

print("Diagnostics generated:")
for k, v in diagnostics.items():
    print(k, "->", len(v), "groups")


Diagnostics generated:
LOCATION -> 29 groups
PRICE_TIER -> 19 groups
LANE -> 39 groups
FEE_CHANGE_PCT -> 4 groups
BASELINE_CONVERSION -> 4 groups
BASELINE_VOLUME -> 4 groups
PREVIOUS_BUYFEE -> 4 groups
BASELINE_RPU -> 4 groups
SALE_AMOUNT -> 4 groups
SEASON -> 4 groups
QUARTER -> 4 groups
MONTH -> 11 groups


## 6. Save diagnostic files

In [7]:

file_map = {
    "LOCATION": "01_LOCATION_SUCCESS_DIAGNOSTICS.csv",
    "PRICE_TIER": "02_PRICE_TIER_SUCCESS_DIAGNOSTICS.csv",
    "LANE": "03_LANE_SUCCESS_DIAGNOSTICS.csv",
    "FEE_CHANGE_PCT": "04_FEE_CHANGE_PCT_DIAGNOSTICS.csv",
    "BASELINE_CONVERSION": "05_BASELINE_CONVERSION_DIAGNOSTICS.csv",
    "BASELINE_VOLUME": "06_BASELINE_VOLUME_DIAGNOSTICS.csv",
    "PREVIOUS_BUYFEE": "07_PREVIOUS_BUYFEE_DIAGNOSTICS.csv",
    "BASELINE_RPU": "08_BASELINE_RPU_DIAGNOSTICS.csv",
    "SALE_AMOUNT": "09_SALE_AMOUNT_DIAGNOSTICS.csv",
    "SEASON": "10_SEASON_DIAGNOSTICS.csv",
    "QUARTER": "11_QUARTER_DIAGNOSTICS.csv",
    "MONTH": "12_MONTH_DIAGNOSTICS.csv",
}

for key, filename in file_map.items():
    diagnostics[key].to_csv(
        OUTPUT_DIR / filename,
        index=False
    )

print("Saved all diagnostic tables.")


Saved all diagnostic tables.


## 7. Successful vs unsuccessful profile

In [8]:

profile_features = [
    "BUYFEE_CHANGE_AMOUNT",
    "FEE_CHANGE_PCT",
    "PREVIOUS_BUYFEE",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

profile_rows = []

for feature in profile_features:

    success = pd.to_numeric(
        df.loc[
            df["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    failure = pd.to_numeric(
        df.loc[
            ~df["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    profile_rows.append({
        "Feature": feature,

        "Success_N": len(success),
        "Failure_N": len(failure),

        "Success_Median": (
            success.median()
            if len(success)
            else np.nan
        ),

        "Failure_Median": (
            failure.median()
            if len(failure)
            else np.nan
        ),

        "Median_Difference": (
            success.median() - failure.median()
            if len(success) and len(failure)
            else np.nan
        ),

        "Success_Mean": (
            success.mean()
            if len(success)
            else np.nan
        ),

        "Failure_Mean": (
            failure.mean()
            if len(failure)
            else np.nan
        ),
    })

success_failure_profile = pd.DataFrame(
    profile_rows
)

success_failure_profile.to_csv(
    OUTPUT_DIR / "13_SUCCESS_VS_FAILURE_PROFILE.csv",
    index=False
)

display(success_failure_profile)


,Feature,Success_N,Failure_N,Success_Median,Failure_Median,Median_Difference,Success_Mean,Failure_Mean
0,BUYFEE_CHANGE_AMOUNT,510,1314,20.000000,20.000000,0.000000,32.210784,33.076484
1,FEE_CHANGE_PCT,510,1314,0.060606,0.060075,0.000531,0.089524,0.095787
2,PREVIOUS_BUYFEE,510,1314,375.000000,365.000000,10.000000,365.777451,355.535388
3,pre_registered,510,1314,48.000000,51.000000,-3.000000,57.764706,61.657534
4,pre_sold,510,1314,27.500000,31.000000,-3.500000,34.652941,37.175038
5,pre_conversion,510,1314,0.600000,0.602869,-0.002869,0.588424,0.599642
6,pre_rpu,510,1314,308.322368,296.592437,11.729931,283.972632,253.435570
7,pre_avg_saleamt,510,1314,5348.255814,4429.696970,918.558844,6562.100186,6061.607815


## 8. Driver ranking by spread in Balanced Success Rate

In [9]:

driver_rows = []

for driver, table in diagnostics.items():

    if table.empty:
        continue

    driver_rows.append({
        "Driver": driver,
        "Groups_Compared": len(table),
        "Best_Balanced_Success_Rate": table["Balanced_Success_Rate"].max(),
        "Worst_Balanced_Success_Rate": table["Balanced_Success_Rate"].min(),
        "Success_Rate_Spread": (
            table["Balanced_Success_Rate"].max()
            - table["Balanced_Success_Rate"].min()
        ),
        "Events_Represented": table["Events"].sum()
    })

driver_ranking = pd.DataFrame(
    driver_rows
).sort_values(
    "Success_Rate_Spread",
    ascending=False
)

driver_ranking.to_csv(
    OUTPUT_DIR / "14_DRIVER_DIAGNOSTIC_RANKING.csv",
    index=False
)

display(driver_ranking)


,Driver,Groups_Compared,Best_Balanced_Success_Rate,Worst_Balanced_Success_Rate,Success_Rate_Spread,Events_Represented
2,LANE,39,0.615385,0.071429,0.543956,1605
0,LOCATION,29,0.482759,0.000000,0.482759,1789
1,PRICE_TIER,19,0.470588,0.080000,0.390588,1785
11,MONTH,11,0.500000,0.150538,0.349462,1816
9,SEASON,4,0.356037,0.211382,0.144655,1824
10,QUARTER,4,0.381974,0.244466,0.137508,1824
7,BASELINE_RPU,4,0.326797,0.192560,0.134237,1824
3,FEE_CHANGE_PCT,4,0.329004,0.231111,0.097893,1824
5,BASELINE_VOLUME,4,0.319239,0.230233,0.089006,1824
8,SALE_AMOUNT,4,0.300439,0.239035,0.061404,1824


## 9. Scenario-specific diagnostics for +$10 / +$15 / +$20 / +$25 / +$30

In [10]:

scenario_rows = []

for fee in [10, 15, 20, 25, 30]:

    d = df[
        df["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    if d.empty:
        continue

    success = d[
        d["Balanced_Success"]
    ]

    failure = d[
        ~d["Balanced_Success"]
    ]

    scenario_rows.append({
        "Fee_Increase": fee,
        "Events": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Balanced_Success_Rate": d["Balanced_Success"].mean(),
        "RPU_Success_Rate": d["RPU_Success"].mean(),
        "Conversion_Safe_Rate": d["Conversion_Safe"].mean(),
        "Volume_Safe_Rate": d["Volume_Safe"].mean(),

        "Median_RPU_Effect": d["DiD_RPU_Change_Dollar"].median(),
        "Median_Conversion_Effect_pp": d["DiD_Conversion_Change_pp"].median(),
        "Median_Volume_Effect_Pct": d["DiD_Volume_Change_Pct"].median(),

        "Successful_Median_Baseline_Conversion": (
            success["pre_conversion"].median()
            if len(success)
            else np.nan
        ),

        "Failed_Median_Baseline_Conversion": (
            failure["pre_conversion"].median()
            if len(failure)
            else np.nan
        ),

        "Successful_Median_Baseline_Sold": (
            success["pre_sold"].median()
            if len(success)
            else np.nan
        ),

        "Failed_Median_Baseline_Sold": (
            failure["pre_sold"].median()
            if len(failure)
            else np.nan
        ),

        "Successful_Median_Previous_BuyFee": (
            success["PREVIOUS_BUYFEE"].median()
            if len(success)
            else np.nan
        ),

        "Failed_Median_Previous_BuyFee": (
            failure["PREVIOUS_BUYFEE"].median()
            if len(failure)
            else np.nan
        ),
    })

scenario_diag = pd.DataFrame(
    scenario_rows
)

scenario_diag.to_csv(
    OUTPUT_DIR / "15_SCENARIO_SUCCESS_DIAGNOSTICS.csv",
    index=False
)

display(scenario_diag)


,Fee_Increase,Events,Locations,Corporate_Change_Dates,Balanced_Success_Rate,RPU_Success_Rate,Conversion_Safe_Rate,Volume_Safe_Rate,Median_RPU_Effect,Median_Conversion_Effect_pp,Median_Volume_Effect_Pct,Successful_Median_Baseline_Conversion,Failed_Median_Baseline_Conversion,Successful_Median_Baseline_Sold,Failed_Median_Baseline_Sold,Successful_Median_Previous_BuyFee,Failed_Median_Previous_BuyFee
0,10,594,16,24,0.289562,0.594276,0.585859,0.611111,7.630647,0.627338,-0.003620,0.637022,0.614482,33.0,32.0,360.0,365.0
1,15,84,14,10,0.297619,0.630952,0.607143,0.654762,10.481982,1.790793,0.044703,0.690476,0.583333,38.0,31.0,339.0,310.0
2,20,133,12,10,0.345865,0.759398,0.586466,0.593985,24.466728,1.658328,-0.025353,0.601587,0.700000,24.5,32.0,375.0,380.0
3,25,42,10,4,0.333333,0.571429,0.619048,0.690476,2.220601,0.901881,0.073548,0.538158,0.626122,24.5,35.5,335.0,290.0
4,30,60,8,4,0.266667,0.550000,0.533333,0.716667,2.512057,-0.547817,0.036691,0.693042,0.559532,22.5,35.0,410.0,310.0


## 10. Final executive summary

In [11]:

summary_rows = [
    {
        "Metric": "Events",
        "Value": len(df)
    },
    {
        "Metric": "Locations",
        "Value": df["Treated_Location"].nunique()
    },
    {
        "Metric": "Corporate_Change_Dates",
        "Value": df["Corporate_Change_ID"].nunique()
    },
    {
        "Metric": "Balanced_Success_Rate",
        "Value": df["Balanced_Success"].mean()
    },
    {
        "Metric": "RPU_Success_Rate",
        "Value": df["RPU_Success"].mean()
    },
    {
        "Metric": "Conversion_Safe_Rate",
        "Value": df["Conversion_Safe"].mean()
    },
    {
        "Metric": "Volume_Safe_Rate",
        "Value": df["Volume_Safe"].mean()
    },
]

executive = pd.DataFrame(
    summary_rows
)

executive.to_csv(
    OUTPUT_DIR / "16_V10_1_EXECUTIVE_SUMMARY.csv",
    index=False
)

df.to_csv(
    OUTPUT_DIR / "17_V10_1_EVENT_LEVEL_WITH_FLAGS.csv",
    index=False
)

print("=" * 80)
print("V10.1 COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(executive)


V10.1 COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v10_1_outputs


,Metric,Value
0,Events,1824.000000
1,Locations,35.000000
2,Corporate_Change_Dates,37.000000
3,Balanced_Success_Rate,0.279605
4,RPU_Success_Rate,0.564145
5,Conversion_Safe_Rate,0.589912
6,Volume_Safe_Rate,0.623355



## Files to share back

Please send:

1. `16_V10_1_EXECUTIVE_SUMMARY.csv`
2. `14_DRIVER_DIAGNOSTIC_RANKING.csv`
3. `13_SUCCESS_VS_FAILURE_PROFILE.csv`
4. `15_SCENARIO_SUCCESS_DIAGNOSTICS.csv`
5. `01_LOCATION_SUCCESS_DIAGNOSTICS.csv`
6. `02_PRICE_TIER_SUCCESS_DIAGNOSTICS.csv`
7. `04_FEE_CHANGE_PCT_DIAGNOSTICS.csv`
8. `05_BASELINE_CONVERSION_DIAGNOSTICS.csv`
9. `06_BASELINE_VOLUME_DIAGNOSTICS.csv`
10. `10_SEASON_DIAGNOSTICS.csv`

This notebook intentionally excludes buyer concentration for stability. Once this runs successfully, buyer concentration can be added in a separate, small enrichment notebook.
